In [7]:
import numpy as np
import torch

from src.process import RFQ

In [8]:
def scurve(delta, alpha, beta, delta_0):
    return 1 / (1 + torch.exp(alpha + beta / delta_0 * delta))


def utility(
    ask_delta,
    bid_delta,
    q,
    z,
    ask_lamda,
    bid_lamda,
    gamma,
    alpha,
    beta,
    delta_0,
    kappa,
    delta_t,
    sigma,
):
    t = ask_delta.shape[1]

    res = 0
    for i in range(t):
        res += (
            z
            * ask_delta[:, i]
            * ask_lamda[:, i]
            * scurve(ask_delta[:, i], alpha, beta, delta_0)
            + z
            * bid_delta[:, i]
            * bid_lamda[:, i]
            * scurve(bid_delta[:, i], alpha, beta, delta_0)
            + kappa * (ask_lamda[:, i] - bid_lamda[:, i]) * q[:, i]
            - gamma / 2 * q[:, i] ** 2 * sigma**2
        ) * delta_t
    return torch.mean(res)

In [9]:
from src.model import RLNet


def fair_price(
    rfq, gamma, delta_0, z, alpha, beta, batch_size=100, epoch=100, lr=0.0001
):
    ask_model = RLNet(5, 1, [10, 20, 10])
    bid_model = RLNet(5, 1, [10, 20, 10])

    ask_optimzer = torch.optim.Adam(ask_model.parameters(), lr=lr, betas=(0.9, 0.99))
    bid_optimizer = torch.optim.Adam(bid_model.parameters(), lr=lr, betas=(0.9, 0.99))

    for _ in range(epoch):
        ask_lamda, bid_lamda, prices = rfq.sample(batch_size)

        ask_lamda = torch.tensor(ask_lamda).to(torch.float32)
        bid_lamda = torch.tensor(bid_lamda).to(torch.float32)
        prices = torch.tensor(prices).to(torch.float32)

        q = torch.zeros((batch_size, rfq.num_time_interval + 1))
        ask_delta = torch.zeros((batch_size, rfq.num_time_interval))
        bid_delta = torch.zeros((batch_size, rfq.num_time_interval))

        for t in range(rfq.num_time_interval):
            ask_optimzer.zero_grad()
            bid_optimizer.zero_grad()

            q[:, t + 1] = z * (bid_lamda[:, t] - ask_lamda[:, t]) * rfq.delta_t
            
            model_input = torch.cat(
                (
                    torch.ones((batch_size, 1)) * t * rfq.delta_t,
                    q[:, t : t + 1],
                    ask_lamda[:, t : t + 1],
                    bid_lamda[:, t : t + 1],
                    prices[:, t : t + 1],
                ),
                dim=1,
            )

            ask_delta[:, t : t + 1] = ask_model(model_input)
            bid_delta[:, t : t + 1] = bid_model(model_input)

        loss = utility(
            ask_delta,
            bid_delta,
            q,
            z,
            ask_lamda,
            bid_lamda,
            gamma,
            alpha,
            beta,
            delta_0,
            rfq.k,
            rfq.delta_t,
            rfq.sigma,
        )
        loss.backward()
        ask_optimzer.step()
        bid_optimizer.step()

        print(f"Loss: {round(loss.item(), 3)}")

In [10]:
basic = {"dim": 1, "total_time": 30, "num_time_interval": 30}
specific = {
    "init": 103.593,  # initial mid-price for bond 1 in sector 1 from Table 3, page 23 in the Bergault paper

    # sigma and kappa from Table 2, page 23 in the Bergault paper
    "sigma": 0.1839 / np.sqrt(252),
    "k": 2.29,

    "nls": 2,
    "init_state": 2,

    # lambda and Q values from Table 1, page 20 in the Bergault paper
    "lamdas": np.array([10.83, 73.03]) / 10, 
    "Q": np.array([
        [-14.01, 4.37, 4.37, 5.27],
        [19.32, -60.91, 12.54, 29.05],
        [19.32, 12.54, -60.91, 29.05],
        [23.67, 15.00, 15.00, -53.67],
    ]),
}

rfq = RFQ(basic, specific)
print(rfq)

RFQ:
{
dim: 
1, 

total_time: 
30, 

num_time_interval: 
30, 

delta_t: 
1.0, 

sqrt_delta_t: 
1.0, 

y_init: 
None, 

n_liqiudity_state: 
2, 

x_init: 
[103.593], 

lamda_initial_state: 
2, 

sigma: 
0.011584611097732815, 

k: 
2.29, 

lamdas: 
[1.083 7.303], 

Q: 
[[-14.01   4.37   4.37   5.27]
 [ 19.32 -60.91  12.54  29.05]
 [ 19.32  12.54 -60.91  29.05]
 [ 23.67  15.    15.   -53.67]]
}


In [11]:
rfq.sample(10)[1].shape

(10, 30)

In [12]:
gamma = 0.0005
z = 1

# s-curve parameters from page 27 of Bergault paper
delta_0 = 0.09
alpha = -0.7
beta = 3.1

fair_price(rfq, gamma, delta_0, z, alpha, beta)

Loss: -32.414
Loss: -25.382
